In [88]:
import numpy as np
import pandas as pd
import requests

In [89]:
BASE_URL = "https://leaderboard-project1-runai-innovation-thibaut.inference.compute.datascience.ch/api/v1"
TEMPLATE_PATH = "/home/renku/work/input-data/Project 1 - DTB/bioreaction_template.csv"
SUBMISSION_PATH = "submissions/blend_kalman4_substrate_only.csv"   # predictions to submit (same rows as the template)
TARGETS = ["vcd", "tcd", "substrate", "byproduct"]


def get_leaderboard() -> pd.DataFrame:
    leaderboard = requests.get(f"{BASE_URL}/leaderboard").json()
    return pd.json_normalize(leaderboard["entries"])


def submit(df: pd.DataFrame, user: str, token: str = "") -> dict:
    submission = {"predictions": df.to_dict(orient="records")}
    response = requests.post(f"{BASE_URL}/submit", json=submission, auth=(user, token))
    if (response.status_code // 100) != 2:
        raise RuntimeError(f"Unexpected error (HTTP {response.status_code}): {response.text}")
    return response.json()["metrics"]

In [90]:
get_leaderboard()

,user_name,rank,attempt,timestamp,metrics.vcd_mape,metrics.tcd_mape,metrics.substrate_mape,metrics.byproduct_mape,metrics.mean_mape
0,Team 7,1,11,2026-11-22T11:33:02.194610,6.627,6.569,6.326,7.314,6.709
1,Team 04,2,16,2026-11-22T10:56:05.255107,6.747,6.550,6.633,6.969,6.725
2,Team 15,3,8,2026-11-22T08:03:13.449316,6.609,6.463,6.479,7.457,6.752
3,Team 23,4,28,2026-11-22T11:05:30.710962,6.534,6.439,6.891,8.256,7.030
4,Team 6,5,14,2026-11-22T11:17:18.903853,6.817,6.588,6.873,8.260,7.135
5,Team 01,6,6,2026-11-22T08:50:00.730814,6.773,6.734,7.389,8.281,7.294
6,Team 10,7,113,2026-11-22T11:24:58.248494,7.164,7.044,7.349,7.644,7.300
7,Team 18,8,17,2026-11-22T11:05:32.839893,7.000,6.900,9.275,8.359,7.883
8,TEMPLATE,9,65,2026-11-21T14:24:40.616352,8.037,7.949,12.023,10.059,9.517


In [91]:
template = pd.read_csv(TEMPLATE_PATH, dtype={"batch_id": str})
df = pd.read_csv(SUBMISSION_PATH, dtype={"batch_id": str})
# Sanity checks: same rows / order as the template, no missing or negative values
assert (df[["batch_id", "elapsed_time"]].values == template[["batch_id", "elapsed_time"]].values).all()
assert df[TARGETS].notna().all().all() and (df[TARGETS] > 0).all().all()
metrics = submit(df[["batch_id", "elapsed_time", *TARGETS]], user="Team 10")
metrics

{'vcd_mape': 7.164,
 'tcd_mape': 7.044,
 'substrate_mape': 7.706,
 'byproduct_mape': 7.644,
 'mean_mape': 7.389}

In [92]:
get_leaderboard()

,user_name,rank,attempt,timestamp,metrics.vcd_mape,metrics.tcd_mape,metrics.substrate_mape,metrics.byproduct_mape,metrics.mean_mape
0,Team 7,1,11,2026-11-22T11:33:02.194610,6.627,6.569,6.326,7.314,6.709
1,Team 04,2,16,2026-11-22T10:56:05.255107,6.747,6.550,6.633,6.969,6.725
2,Team 15,3,8,2026-11-22T08:03:13.449316,6.609,6.463,6.479,7.457,6.752
3,Team 23,4,28,2026-11-22T11:05:30.710962,6.534,6.439,6.891,8.256,7.030
4,Team 6,5,14,2026-11-22T11:17:18.903853,6.817,6.588,6.873,8.260,7.135
5,Team 01,6,6,2026-11-22T08:50:00.730814,6.773,6.734,7.389,8.281,7.294
6,Team 10,7,114,2026-11-22T11:24:58.248494,7.164,7.044,7.349,7.644,7.300
7,Team 18,8,17,2026-11-22T11:05:32.839893,7.000,6.900,9.275,8.359,7.883
8,TEMPLATE,9,65,2026-11-21T14:24:40.616352,8.037,7.949,12.023,10.059,9.517
